##Mining Software Repositories Assignment: Human–AI Collaboration Study - LOG6307E A2026

RQ1: How do merge rates differ between AI-authored and human-authored pull requests?

RQ2: How does the size of an Agentic pull request relate to its likelihood of being merged?

## 1. Environment setup

In [3]:
!pip install -q requests pandas pyarrow fastparquet matplotlib backoff seaborn cliffs-delta jupyter scipy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 37.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 69.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 947.5/947.5 kB 49.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.5/77.5 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.2/60.2 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 103.9 MB/s eta 0:00:00


## 2. Imports

In [4]:
import pandas as pd

## 3. Dataset

### 3.1. Agent authored pull requests

In [5]:
# loading Agent PRs and required columns
agent_pr = pd.read_parquet(
    "hf://datasets/hao-li/AIDev@v3/pull_request.parquet",
    columns=[
        'id', 'number', 'title', 'body', 'agent',
        'user_id', 'user', 'state',
        'created_at', 'closed_at', 'merged_at',
        'repo_id', 'repo_url', 'html_url'
    ]
)

# create merged column:  True = merged_at contains a date, False = merged_at is missing
agent_pr["merged"] = agent_pr["merged_at"].notna()


# loading file changes and required columns
pr_commit_details = pd.read_parquet(
    "hf://datasets/hao-li/AIDev@v3/pr_commit_details.parquet",
    columns=[
        'pr_id', 'sha', 'filename', 'status',
        'additions', 'deletions', 'changes'
    ]
)

# calculate PR size
pr_size = (
    pr_commit_details
    .groupby("pr_id")
    .agg(
        additions=("additions", "sum"),
        deletions=("deletions", "sum"),
        files_changed=("filename", "nunique")
    )
    .reset_index()
)

# total size of the PR
pr_size["change_size"] = (
    pr_size["additions"] +
    pr_size["deletions"]
)


# loading comments and required columns
pr_comments = pd.read_parquet(
    "hf://datasets/hao-li/AIDev@v3/pr_comments.parquet",
    columns=[
        'pr_id', 'id', 'user', 'user_id',
        'user_type', 'created_at', 'body'
    ]
)


# loading reviews and required columns
pr_reviews = pd.read_parquet(
    "hf://datasets/hao-li/AIDev@v3/pr_reviews.parquet",
    columns=[
        'pr_id', 'id', 'user', 'user_type',
        'state', 'submitted_at', 'body'
    ]
)



agent_pr["number_of_comments"] = (
    agent_pr["id"]
    .map(pr_comments["pr_id"].value_counts())
    .fillna(0)
    .astype(int)
)

agent_pr["number_of_reviews"] = (
    agent_pr["id"]
    .map(pr_reviews["pr_id"].value_counts())
    .fillna(0)
    .astype(int)
)


# validations
print("Agent PRs:", agent_pr.shape)
print("Commit details:", pr_commit_details.shape)
print("PR size:", pr_size.shape)
print("Comments:", pr_comments.shape)
print("Reviews:", pr_reviews.shape)

print(agent_pr[["id", "merged_at", "merged"]].head())
print(pr_size.head())

Agent PRs: (33596, 17)
Commit details: (711923, 7)
PR size: (33580, 5)
Comments: (39122, 7)
Reviews: (28875, 7)
           id             merged_at  merged
0  3264933329                  None   False
1  3265118634  2025-07-26T22:12:24Z    True
2  3265640341  2025-07-26T13:37:22Z    True
3  3265709660  2025-07-26T14:45:30Z    True
4  3265782173                  None   False
        pr_id  additions  deletions  files_changed  change_size
0  2756921963      848.0      344.0             15       1192.0
1  2757103560      517.0      262.0             16        779.0
2  2757124156        1.0        1.0              1          2.0
3  2757125491        1.0        1.0              1          2.0
4  2757179026     1633.0      169.0             15       1802.0


In [6]:
#All pr_id values in commit details, comments, and reviews matched an id in agent_pr
check = {
    "commit details": pr_commit_details,
    "comments": pr_comments,
    "reviews": pr_reviews
}

for name, df in check.items():
    unmatched = (~df["pr_id"].isin(agent_pr["id"])).sum()
    print(f"{name}: {unmatched} unmatched rows")

commit details: 0 unmatched rows
comments: 0 unmatched rows
reviews: 0 unmatched rows


### 3.2 Human authored pull requests

In [7]:
# loading Human PRs and required columns
human_pr = pd.read_parquet(
    "hf://datasets/hao-li/AIDev@v3/human_pull_request.parquet",
    columns=[
        'id', 'number', 'title', 'body','agent',
        'user_id', 'user', 'state',
        'created_at', 'closed_at', 'merged_at',
        'repo_url', 'html_url'
    ]
)

# create merged column
human_pr["merged"] = human_pr["merged_at"].notna()

print("Human PRs:", human_pr.shape)

print("Columns:")
print(human_pr.columns.tolist())

print(human_pr[["id", "merged_at", "merged"]].head())

human_pr.head()

Human PRs: (6618, 14)
Columns:
['id', 'number', 'title', 'body', 'agent', 'user_id', 'user', 'state', 'created_at', 'closed_at', 'merged_at', 'repo_url', 'html_url', 'merged']
           id             merged_at  merged
0  2336888723  2025-02-18T22:42:19Z    True
1  2447123365  2025-04-09T15:56:54Z    True
2  2438086945  2025-04-04T15:10:57Z    True
3  2265431531  2025-01-08T08:49:40Z    True
4  2332333882  2025-02-12T22:20:33Z    True


,id,number,title,body,agent,user_id,user,state,created_at,closed_at,merged_at,repo_url,html_url,merged
0,2336888723,85268,feat(aci): add automations index page,https://sentry-j41gpomr5.sentry.dev/automation...,Human,55610339,ameliahsu,closed,2025-02-14T19:04:59Z,2025-02-18T22:42:20Z,2025-02-18T22:42:19Z,https://api.github.com/repos/getsentry/sentry,https://github.com/getsentry/sentry/pull/85268,True
1,2447123365,89131,ref(insights): Make use of `<FeatureBadge>` fo...,Using the premade component reduces an import ...,Human,187460,ryan953,closed,2025-04-08T23:29:50Z,2025-04-09T15:56:55Z,2025-04-09T15:56:54Z,https://api.github.com/repos/getsentry/sentry,https://github.com/getsentry/sentry/pull/89131,True
2,2438086945,88748,:bug: fix: update how we fetch workflow_id and...,i realized i made a mistake for how i fetch th...,Human,33237075,iamrajjoshi,closed,2025-04-03T21:36:59Z,2025-04-04T15:10:57Z,2025-04-04T15:10:57Z,https://api.github.com/repos/getsentry/sentry,https://github.com/getsentry/sentry/pull/88748,True
3,2265431531,83085,fix(org-stats): Require project membership,### Problem\r\n\r\nIf the user is not member o...,Human,7033940,ArthurKnaus,closed,2025-01-08T07:47:13Z,2025-01-08T08:49:40Z,2025-01-08T08:49:40Z,https://api.github.com/repos/getsentry/sentry,https://github.com/getsentry/sentry/pull/83085,True
4,2332333882,85102,ref(consumers): Rename parallel -> batched-par...,Both crons and uptime consumers have a paralle...,Human,1421724,evanpurkhiser,closed,2025-02-12T21:24:17Z,2025-02-12T22:20:33Z,2025-02-12T22:20:33Z,https://api.github.com/repos/getsentry/sentry,https://github.com/getsentry/sentry/pull/85102,True


### 3.3 overlap agent and human authored PRs (based on html_url) TO EXCLUDE FROM DATASET DURING PART 2 (see Anthonia's answer below)
Note for part 2:
- 56 PRs are in both Agent and Human datasets based on the same html_url
- 0 PR ids overlap between the Agent and Human datasets
- check why they are in both groups and decide on how to handle these overlaps before calculating merge rates.
- Also 5 of them have conflicting merge status between agent and human dataset

I asked Anthonia if this is expected or if it should be treated as a data issue. This is her answer: **I think that removing them completely from the datasets will not wound your analysis since we have more PRs and we are unsure about the quality of collection of that human PR dataset.**



In [8]:
# check overlap based on PR id
id_overlap = set(agent_pr["id"]).intersection(set(human_pr["id"]))

print("Number of overlapping PR ids:", len(id_overlap))

Number of overlapping PR ids: 0


In [9]:
#list of overlaps between agent and human pr
overlap = agent_pr.merge(
    human_pr,
    on="html_url",
    how="inner",
    suffixes=("_agent", "_human")
)

print("Number of overlapping PR URLs:", len(overlap))

overlap[
    [
        "html_url",
        "id_agent",
        "user_agent",
        "agent_agent",
        "merged_agent",
        "id_human",
        "user_human",
        "agent_human",
        "merged_human"
    ]
].head(20)


Number of overlapping PR URLs: 56


,html_url,id_agent,user_agent,agent_agent,merged_agent,id_human,user_human,agent_human,merged_human
0,https://github.com/liam-hq/liam/pull/2156,3167450477,MH4GF,Claude_Code,True,2611004920,MH4GF,Human,True
1,https://github.com/microsoft/azure-pipelines-a...,3172683435,Copilot,Copilot,False,2615638345,Copilot,Human,False
2,https://github.com/microsoft/vscode/pull/252336,3173280313,Copilot,Copilot,False,2616159930,Copilot,Human,False
3,https://github.com/dotnet/aspire/pull/10027,3173531582,Copilot,Copilot,False,2616384311,Copilot,Human,False
4,https://github.com/microsoft/vscode-mssql/pull...,3176615444,Copilot,Copilot,False,2619066635,Copilot,Human,False
5,https://github.com/featureform/enrichmcp/pull/90,3172714215,simba-git,OpenAI_Codex,True,2615665027,simba-git,Human,True
6,https://github.com/katspaugh/wavesurfer.js/pul...,3171969055,katspaugh,OpenAI_Codex,True,2615010492,katspaugh,Human,True
7,https://github.com/bionic-gpt/bionic-gpt/pull/880,3172047870,242816,OpenAI_Codex,False,2615080183,242816,Human,False
8,https://github.com/Codehagen/Badget/pull/292,3178228056,Codehagen,OpenAI_Codex,True,2620435217,Codehagen,Human,False
9,https://github.com/bionic-gpt/bionic-gpt/pull/879,3168637428,242816,OpenAI_Codex,True,2612067690,242816,Human,True


In [10]:
#list of overlaps with conflicting merge status between agent and human pr
overlap["merge_status_conflict"] = (
    overlap["merged_agent"] != overlap["merged_human"]
)

overlap[
    [
        "html_url",
        "agent_agent",
        "user_agent",
        "merged_agent",
        "agent_human",
        "user_human",
        "merged_human",
        "merge_status_conflict"
    ]
].head(10)

,html_url,agent_agent,user_agent,merged_agent,agent_human,user_human,merged_human,merge_status_conflict
0,https://github.com/liam-hq/liam/pull/2156,Claude_Code,MH4GF,True,Human,MH4GF,True,False
1,https://github.com/microsoft/azure-pipelines-a...,Copilot,Copilot,False,Human,Copilot,False,False
2,https://github.com/microsoft/vscode/pull/252336,Copilot,Copilot,False,Human,Copilot,False,False
3,https://github.com/dotnet/aspire/pull/10027,Copilot,Copilot,False,Human,Copilot,False,False
4,https://github.com/microsoft/vscode-mssql/pull...,Copilot,Copilot,False,Human,Copilot,False,False
5,https://github.com/featureform/enrichmcp/pull/90,OpenAI_Codex,simba-git,True,Human,simba-git,True,False
6,https://github.com/katspaugh/wavesurfer.js/pul...,OpenAI_Codex,katspaugh,True,Human,katspaugh,True,False
7,https://github.com/bionic-gpt/bionic-gpt/pull/880,OpenAI_Codex,242816,False,Human,242816,False,False
8,https://github.com/Codehagen/Badget/pull/292,OpenAI_Codex,Codehagen,True,Human,Codehagen,False,True
9,https://github.com/bionic-gpt/bionic-gpt/pull/879,OpenAI_Codex,242816,True,Human,242816,True,False


In [11]:
## Mount Drive in the Part 1 notebook
from google.colab import drive
import os

drive.mount('/content/drive')

PROJECT_DIR = '/content/drive/MyDrive/LOG6307E_project'
RAW_DIR = os.path.join(PROJECT_DIR, 'data', 'raw')
os.makedirs(RAW_DIR, exist_ok=True)
print("Saving to:", RAW_DIR)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Saving to: /content/drive/MyDrive/LOG6307E_project/data/raw


In [12]:
## Save the raw tables
# Raw data as mined in Part 1, no cleaning applied yet
agent_pr.to_parquet(os.path.join(RAW_DIR, 'agent_pr.parquet'), index=False)
human_pr.to_parquet(os.path.join(RAW_DIR, 'human_pr.parquet'), index=False)
pr_commit_details.to_parquet(os.path.join(RAW_DIR, 'pr_commit_details.parquet'), index=False)
pr_size.to_parquet(os.path.join(RAW_DIR, 'pr_size.parquet'), index=False)

# URLs found in both agent and human sets (to be removed in Part 2)
overlap[['html_url', 'merged_agent', 'merged_human']].to_csv(
    os.path.join(RAW_DIR, 'overlap_urls.csv'), index=False
)

In [13]:
## Verify the save
for f in sorted(os.listdir(RAW_DIR)):
    size_mb = os.path.getsize(os.path.join(RAW_DIR, f)) / 1e6
    print(f"{f}: {size_mb:.1f} MB")

agent_pr.parquet: 16.7 MB
human_pr.parquet: 4.4 MB
overlap_urls.csv: 0.0 MB
pr_commit_details.parquet: 16.9 MB
pr_size.parquet: 0.5 MB
